Este codigo contiene una funcion que repite un experimento de inferencia montecarlo_idx-veces con los parametros que el usuario elija y mide los errores en norma, en soporte y la esparcidad media de todas las iteraciones.

In [1]:
from sim_senales_y_matrices import simular,senal
from optimizacion import algoritmo_proximal_lasso
from utils import distancias
from medir_error import error_sop
import numpy as np
def ejecutar_una_iteracion_mc(args):
        """Ejecuta una única simulación de Montecarlo de forma aislada."""
        montecarlo_idx, k, m, n, alpha, beta, lambd, a0, iters, l, epsilon = args
        np.random.seed()
        matrices_sop_comun = simular(m, k)
        matrices_sop_comun = [w / np.max(np.abs(w)) for w in matrices_sop_comun] 
        senales = [senal(G, m, n) for G in matrices_sop_comun]
        
        normalizador = 1
        matrices_distancia = [(1 / normalizador) * distancias(s, m) for s in senales]
        matrices_init = [np.ones(l) for init_idx in range(0, k)]
        
        W_conj, valores = algoritmo_proximal_lasso(
            matrices_init, matrices_distancia, alpha, beta, lambd, a0, iters, 
            armijo=True, c=0.1, tao=0.1, iter_reporte=[], epsilon = epsilon
        )
        
        err_sop = np.mean([error_sop(W_conj[i], matrices_sop_comun[i]) for i in range(0, k)])
    
        W_conj = [w / np.max(np.abs(w)) for w in W_conj]
        temp1_norm = 0  
        for i in range(0, k):
            temp1_norm += np.linalg.norm(W_conj[i] - matrices_sop_comun[i])
    
        
        
        err_mc_norm = temp1_norm / k
        esparci = [sum([w!=0 for w in W_conj[i]]) for i in range(0,k)]
        
        return err_mc_norm,err_sop, esparci

In [2]:
m = 10 # cantidad de nodos
k = 2 #cantidad de grafos inferidos simultaneamente
n = 10 #largo de las señales que se generará
l = m*(m-1)//2 


alpha = 1 # peso del regularizador termino log
beta = 1 #peso del regularizador termino norma 2 al cuadrado
a0 = 1 # peso del regularizador que obliga a las matrices a compartir soporte

lambd = 0.0001 #tamaño del paso de descenso por gradiente (gamma en la tesis)

iters = 5000 #cantidad maxima de iteraciones si no se cumple la condicion de parada
epsilon = 0.0001 #espilon de la condicion de parada || T(x) -x||_2 < \epsilon 

montecarlo_idx = 10 #cantidad de repeticiones del experimento.


args = montecarlo_idx, k, m, n, alpha, beta, lambd, a0, iters, l, epsilon


In [3]:
error_en_norma ,error_en_soporte , esparcidad_m_inferidas = ejecutar_una_iteracion_mc(args)